In [8]:
# Initialize gee
import geemap
import ee
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [9]:
# Carregar camada
path_json = "../outputs/bacias_meso_SF.geojson"
geom_ee = geemap.geojson_to_ee(path_json)
area = geom_ee.geometry()

In [10]:
# 3. Carregar a coleção MODIS (Reflectância de Superfície)
modis = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2020-01-01', '2020-12-31')

In [11]:
# 4. Função para calcular MSAVI e Albedo
def add_indices(image):
    # Aplicar fator de escala do MODIS para obter a reflectância real [7]
    img_scaled = image.multiply(0.0001) 
    
    # Cálculo do MSAVI usando .expression() [4]
    msavi = img_scaled.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': img_scaled.select('sur_refl_b02'), # Banda NIR no MODIS
            'RED': img_scaled.select('sur_refl_b01')  # Banda RED no MODIS
        }).rename('MSAVI')
        
    # Cálculo do Albedo (Exemplo usando fórmula empírica comum para MODIS)
    # Verifique os coeficientes exatos da metodologia que você está seguindo
    albedo = img_scaled.expression(
        '0.160 * B1 + 0.291 * B2 + 0.243 * B3 + 0.116 * B4 + 0.112 * B5 + 0.081 * B7 - 0.0015', {
            'B1': img_scaled.select('sur_refl_b01'),
            'B2': img_scaled.select('sur_refl_b02'),
            'B3': img_scaled.select('sur_refl_b03'),
            'B4': img_scaled.select('sur_refl_b04'),
            'B5': img_scaled.select('sur_refl_b05'),
            'B7': img_scaled.select('sur_refl_b07')
        }).rename('Albedo')

    # Adiciona as novas bandas à imagem original e mantém as propriedades de data [4, 8]
    return image.addBands([msavi, albedo]).copyProperties(image, ['system:time_start'])

In [15]:
# 5. Mapear a função sobre toda a coleção de imagens
modis_com_indices = modis.map(add_indices)

In [19]:
modis_com_indices

In [20]:
import pandas as pd

# 1. Função para extrair a média e a data de cada imagem
def extrair_serie(image):
    # Calcula a média do MSAVI e Albedo dentro do seu ROI
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area,
        scale=500, # Resolução nativa do MODIS em metros
        maxPixels=1e9
    )
    
    # Retorna os dados como atributos de uma Feature (sem a geometria pesada)
    return ee.Feature(None, {
        'Data': image.date().format('YYYY-MM-dd'),
        'MSAVI': stats.get('MSAVI'),
        'Albedo': stats.get('Albedo')
    })
    

In [ ]:
# 2. Mapeia a função sobre a coleção que criamos no passo anterior
serie_temporal_fc = modis_com_indices.map(extrair_serie)

# 3. Transforma a FeatureCollection do Earth Engine direto para um DataFrame do Pandas
df = geemap.ee_to_df(ee.FeatureCollection(serie_temporal_fc))

# 4. Limpa e organiza a tabela
df = df.dropna().sort_values('Data')
print(df.head())